# Full English Wikipedia offline RAG — Flan-T5-small

This separate notebook indexes **every article in all 41 Parquet shards** of `20231101.en` (11,630,929,031 compressed bytes). There is no 500/2500-article limit. Existing data and model files are reused; the old index is untouched.

Run cells in order. The first index build can take many hours and use tens to hundreds of GB beyond the compressed source. Keep at least 100 GB free initially; the notebook stops below 10 GB free and can resume after space is freed. Actual size depends on passage count. SQLite BM25 uses a disk-backed inverted index; corpus text and postings are not loaded into RAM together. Batch memory is bounded except for the largest individual article. Restart and rerun to resume committed batches.

**Scope:** arbitrary English questions can search the entire indexed snapshot. Coverage is not a guarantee of answer correctness: lexical retrieval can miss paraphrases, Flan-T5-small has limited reasoning and a 512-token context, and the snapshot is from November 2023. This implementation favors short extractive factual answers and abstains when the generated answer is not a span of visible evidence. That check is lexical support, not proof of entailment. Long-form, multi-hop, and exhaustive aggregation questions are not guaranteed.

Offline by default. Initial package installation or missing-asset download requires explicitly setting `ALLOW_DOWNLOADS=True`. No API or hosted inference is used. The notebook is self-contained.


In [43]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter
from contextlib import contextmanager
from uuid import uuid4
import copy
import csv
import hashlib
import importlib.metadata
import importlib.util
import itertools
import json
import os
import platform
import re
import shutil
import socket
import sqlite3
import statistics
import string
import subprocess
import sys
import tempfile
import time
from unittest import mock

PROJECT_DIR = Path("/Users/amolsingh/Documents/code/IITB-DL")
ROOT = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR is not None else Path.cwd().resolve()
DATASET_DIR = ROOT / "data" / "wikipedia_snapshot"
DOWNLOAD_SCOPE = "english"  # The complete English subset only.
MODEL_DIR = ROOT / "models" / "flan-t5-small"
NOTEBOOK_NAME = "offline_wikipedia_rag_full.ipynb"
ALLOW_DOWNLOADS = False
INSTALL_MISSING_PACKAGES = True
VERIFY_DATASET_HASHES = False
DOWNLOAD_WORKERS = 4
QUESTION = "What is the atomic number of aluminium?"
K = 3
KS = [1, 3, 5]
#RUN_FULL_ABLATION = True
RUN_FULL_ABLATION = False
DEVICE = "cpu"
THREADS = 4
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 48

DATASET = "wikimedia/wikipedia"
DATASET_REVISION = "b04c8d1ceb2f5cd4588862100d08de323dccfbaa"
CONFIG = "20231101.en"
MODEL = "google/flan-t5-small"
MODEL_REVISION = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"
ROOT.mkdir(parents=True, exist_ok=True)
print("Working directory:", ROOT)
print("Complete English download:", DATASET_DIR)
print("Python:", sys.executable)


INDEX_PATH = ROOT / "data" / "full_wikipedia_bm25.sqlite"
CHUNK_WORDS, OVERLAP, BATCH_SIZE = 100, 25, 128
MIN_FREE_GB = 10


Working directory: /Users/amolsingh/Documents/code/IITB-DL
Complete English download: /Users/amolsingh/Documents/code/IITB-DL/data/wikipedia_snapshot
Python: /opt/homebrew/Caskroom/miniforge/base/bin/python


## Dependencies and pinned local assets
Existing installations are reused. Model and dataset inventories below are inherited from the original notebook. Default validation checks dataset sizes; enable hashes for a slower full integrity check.

In [44]:
PACKAGE_REQUIREMENTS = {'torch': 'torch==2.12.0', 'transformers': 'transformers==4.57.6', 'huggingface_hub': 'huggingface-hub==0.36.2', 'sentencepiece': 'sentencepiece==0.2.1', 'pyarrow': 'pyarrow==26.0.0', 'numpy': 'numpy==2.4.2', 'pandas': 'pandas==3.0.0', 'matplotlib': 'matplotlib==3.10.8'}
missing = [requirement for name, requirement in PACKAGE_REQUIREMENTS.items()
           if importlib.util.find_spec(name) is None]
if missing:
    if not (INSTALL_MISSING_PACKAGES and ALLOW_DOWNLOADS):
        raise RuntimeError("Install these packages while online: " + ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required Python packages are already installed; installation skipped.")

import pandas as pd
from IPython.display import Image, Markdown, display


All required Python packages are already installed; installation skipped.


In [45]:
EMBEDDED_INVENTORY = {'schema_version': 1,
 'dataset': {'repo_id': 'wikimedia/wikipedia',
             'repo_type': 'dataset',
             'revision': 'b04c8d1ceb2f5cd4588862100d08de323dccfbaa',
             'files': [{'path': '.gitattributes',
                        'size': 2307,
                        'git_sha1': '28df5f900b358436f0267334b3e3e9af33f917ba'},
                       {'path': '20231101.en/train-00000-of-00041.parquet',
                        'size': 420296449,
                        'sha256': '382e7f6f09e488b24793a7f7cfc659879d5a22da2cf2efec6491665f0c019677'},
                       {'path': '20231101.en/train-00001-of-00041.parquet',
                        'size': 351448367,
                        'sha256': '6cd57a5825312ca966313d4a1b614690b5c4fb53cee9b124bc0e999977eb36cf'},
                       {'path': '20231101.en/train-00002-of-00041.parquet',
                        'size': 329024669,
                        'sha256': 'b18674f163d4f106e43396fd58412a79b80a418c72bb728c20af439e7ca4d5b7'},
                       {'path': '20231101.en/train-00003-of-00041.parquet',
                        'size': 330779424,
                        'sha256': '0b8a6f452f5ef59d0986a067f70110feb16d7c1d281ba2db0b3df32933618fcc'},
                       {'path': '20231101.en/train-00004-of-00041.parquet',
                        'size': 306945391,
                        'sha256': '1424300f113c13ee022b7929185fa4f7e20c6195ba563d728e8a3dc47c8e9aba'},
                       {'path': '20231101.en/train-00005-of-00041.parquet',
                        'size': 243974056,
                        'sha256': '220d31bb222ba2d449c845733622a6a2b7a89db3e28b24bfcb3a9bb3e52c6c8d'},
                       {'path': '20231101.en/train-00006-of-00041.parquet',
                        'size': 266288728,
                        'sha256': 'b8becbe9989a2c63f129a707a4674a84fb093c163673b1f4ce97afbf5306f04c'},
                       {'path': '20231101.en/train-00007-of-00041.parquet',
                        'size': 228195804,
                        'sha256': 'adebba8b075444923ba42ed6214fe205efc756c773f440d98cbdd88a33597c2c'},
                       {'path': '20231101.en/train-00008-of-00041.parquet',
                        'size': 247687191,
                        'sha256': '905f959921ab8de07e7ce238d7c9ef33bd597585a17a99c3ba00163b25002bef'},
                       {'path': '20231101.en/train-00009-of-00041.parquet',
                        'size': 227378231,
                        'sha256': '465bd99719b0f2d1974402b7922a7a66399675dc6218a0a40057e0512592c07b'},
                       {'path': '20231101.en/train-00010-of-00041.parquet',
                        'size': 233604795,
                        'sha256': 'cd8b4e9ee2506ba5f47d552c56e2a7569402a6814d9fa171c0e995e56859e51b'},
                       {'path': '20231101.en/train-00011-of-00041.parquet',
                        'size': 231748583,
                        'sha256': '881e8efe6dbaddd34791fdc2158123ed344c436263b9052a0d659418fff89c3a'},
                       {'path': '20231101.en/train-00012-of-00041.parquet',
                        'size': 239222511,
                        'sha256': 'd33835c5e69d5f2464cab1ad70b998ec37e084ea0c0ef9f837fac4a9bb030860'},
                       {'path': '20231101.en/train-00013-of-00041.parquet',
                        'size': 241146961,
                        'sha256': '9abcab8c808952dc6a4822da1591db51de980a792ed37acd3bd243ebd99d200e'},
                       {'path': '20231101.en/train-00014-of-00041.parquet',
                        'size': 222798405,
                        'sha256': '7419912e150b17ce395ea039620c21a7e1cb54731c7c2fd2ccbbed56674eeccd'},
                       {'path': '20231101.en/train-00015-of-00041.parquet',
                        'size': 234578705,
                        'sha256': '1d11ff44d5579dbb9e1976330cdf70ec8b04a7a5dfe4113916b71551fb3cf715'},
                       {'path': '20231101.en/train-00016-of-00041.parquet',
                        'size': 503153575,
                        'sha256': 'e559da89b795e365bc1d1bb87576d1a5478c346cf37eb06159caf9579fe1e54e'},
                       {'path': '20231101.en/train-00017-of-00041.parquet',
                        'size': 230813705,
                        'sha256': '216cd1e4921cc884ebdaed94b61d3e15560f502f73752f3694ed05e6347a6562'},
                       {'path': '20231101.en/train-00018-of-00041.parquet',
                        'size': 231055386,
                        'sha256': '64fb53939ab6aae24299cdafa2ce88e115d91930be35609b2dcfc1fc3b9f85eb'},
                       {'path': '20231101.en/train-00019-of-00041.parquet',
                        'size': 195276969,
                        'sha256': '99c23125b0dd115d68f4a921376880e307bdfcbf3bfd66ae4ed90e853e1dadac'},
                       {'path': '20231101.en/train-00020-of-00041.parquet',
                        'size': 225141733,
                        'sha256': '79ee06627f4ba2fec056fcd23c077a29238bbcbacc3187e8d1a0b47e663e9c7a'},
                       {'path': '20231101.en/train-00021-of-00041.parquet',
                        'size': 216445587,
                        'sha256': 'e9f695672d916793439d8f48893ecde94c61b0eb038a0034c4e1f1f194205979'},
                       {'path': '20231101.en/train-00022-of-00041.parquet',
                        'size': 201995404,
                        'sha256': '50ba80435bf2fd0b818533a1ca3118454a3ae275dc2250e0b68dc852dd03f3d2'},
                       {'path': '20231101.en/train-00023-of-00041.parquet',
                        'size': 212615391,
                        'sha256': 'b722b7e4039c65382286b6aa5a35f9a4c5d3536b758c346a94af7a2a692cc6bb'},
                       {'path': '20231101.en/train-00024-of-00041.parquet',
                        'size': 220729597,
                        'sha256': 'eb9818bff41110829f39f8100aee5936cf6088c72a5925c2b08bcd3c37ac0170'},
                       {'path': '20231101.en/train-00025-of-00041.parquet',
                        'size': 221292721,
                        'sha256': 'fed7d34add9b161745dd2677ff3c5f782a471f433bae80c59933e9b68225be93'},
                       {'path': '20231101.en/train-00026-of-00041.parquet',
                        'size': 208188233,
                        'sha256': 'ffffb7d303ef78b47c4c835211b35f6052ffa2d5998e3f65d69bb7fe512ee69c'},
                       {'path': '20231101.en/train-00027-of-00041.parquet',
                        'size': 213980716,
                        'sha256': '1f0c7bcaf2aeb9765a981c8911b95ab3a9e50456bbad64e50e722a53b8608ccc'},
                       {'path': '20231101.en/train-00028-of-00041.parquet',
                        'size': 188376392,
                        'sha256': '10589a39188af404fa458da252df7dcf22c6f6395a1501d9243ef56ce4c3c148'},
                       {'path': '20231101.en/train-00029-of-00041.parquet',
                        'size': 217775627,
                        'sha256': '8f24c4250d69dbee36962d587cac632098635fa78e8705db140cce68928bb9d3'},
                       {'path': '20231101.en/train-00030-of-00041.parquet',
                        'size': 203945186,
                        'sha256': '1de329963a4aaed0c40d820a564487dc98ef113c8ca6b88b11d495d6a4b98774'},
                       {'path': '20231101.en/train-00031-of-00041.parquet',
                        'size': 215470925,
                        'sha256': '55e58e33710b9326e1ce3aab1319377afd9ce18df089b08aba2ab9286228666e'},
                       {'path': '20231101.en/train-00032-of-00041.parquet',
                        'size': 214242576,
                        'sha256': '354c4d35fa089351ed4ec1622fd8930f309dddd3dccbf411e4dd8a52eff638d4'},
                       {'path': '20231101.en/train-00033-of-00041.parquet',
                        'size': 202706223,
                        'sha256': 'a5140a1c80805d599b791fbd668d0f49972eecf0cd2d863cede53013f52e0900'},
                       {'path': '20231101.en/train-00034-of-00041.parquet',
                        'size': 219010799,
                        'sha256': '298528554e93a3fae9f694087a7376b70be223084b2739ef9488418202a948ab'},
                       {'path': '20231101.en/train-00035-of-00041.parquet',
                        'size': 224459876,
                        'sha256': '460ab2f5a2da091e310d0f86c461a73f22b189f22ac7115e4708a5da9baf4d1c'},
                       {'path': '20231101.en/train-00036-of-00041.parquet',
                        'size': 610276598,
                        'sha256': '7385c9f0482cd592578c1c22c5eaaad8bc3013bec4e3bfe394b441dc728f78a9'},
                       {'path': '20231101.en/train-00037-of-00041.parquet',
                        'size': 674143674,
                        'sha256': 'f2f7718efc3165ed599ec0d99fef07b98ec4dd0da902ffc330dd5c75016240cb'},
                       {'path': '20231101.en/train-00038-of-00041.parquet',
                        'size': 538231796,
                        'sha256': '16ff05ab829cb9c983db14b7f77a677d279f8120571e67b2793735851b71ae89'},
                       {'path': '20231101.en/train-00039-of-00041.parquet',
                        'size': 464851854,
                        'sha256': '63a0be9b1b458b834428c897e37a6e2643e0c33dd4ac08a994f3b3dbeee124e1'},
                       {'path': '20231101.en/train-00040-of-00041.parquet',
                        'size': 421630218,
                        'sha256': 'fed6addb1da9364fcf42c6953539eecc21bbc2cf3cd33743684e6872ce6c468a'},
                       {'path': 'README.md',
                        'size': 130835,
                        'git_sha1': '20cd0049340ebfdf4512171270e3b6ae71a25e8d'}],
             'total_bytes': 11631062173},
 'model': {'repo_id': 'google/flan-t5-small',
           'repo_type': 'model',
           'revision': '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab',
           'files': [{'path': 'README.md',
                      'size': 10820,
                      'git_sha1': '245f4a935cf3b2c65ed3d0c1a11b15fd6ab43b80'},
                     {'path': 'config.json',
                      'size': 1401,
                      'git_sha1': '360a0073f29a105ced4a366cda27e668b11bb73b'},
                     {'path': 'generation_config.json',
                      'size': 147,
                      'git_sha1': 'd52815623b46b7db1c4b957b5a83a8ad30b0146a'},
                     {'path': 'model.safetensors',
                      'size': 307867048,
                      'sha256': '495fa51e204676f1a857a9fc13c4c89f3f5ba9f480b898cebca02add25e6d749'},
                     {'path': 'special_tokens_map.json',
                      'size': 2201,
                      'git_sha1': '2c19eb6e3b583f52d34b903b5978d3d30b6b7682'},
                     {'path': 'spiece.model',
                      'size': 791656,
                      'sha256': 'd60acb128cf7b7f2536e8f38a5b18a05535c9e14c7a355904270e15b0945ea86'},
                     {'path': 'tokenizer.json',
                      'size': 2424064,
                      'git_sha1': 'fc669fbad1a6a82119ca3e1fa75db33ee22ca47d'},
                     {'path': 'tokenizer_config.json',
                      'size': 2539,
                      'git_sha1': 'db13bf98b7714acc4dea7621ff7f4ab93f64258e'}],
           'total_bytes': 311099876}}

EVAL_QUESTIONS = [{'id': 'q01',
  'question': 'What is the atomic number of aluminium?',
  'answerable': True,
  'answers': ['13'],
  'article_id': '904',
  'source_title': 'Aluminium',
  'source_url': 'https://en.wikipedia.org/wiki/Aluminium',
  'evidence_quote': 'is a chemical element with the symbol\xa0Al and atomic '
                    'number\xa013',
  'gold_passage_ids': ['904:0']},
 {'id': 'q02',
  'question': 'What is the capital of Alabama?',
  'answerable': True,
  'answers': ['Montgomery'],
  'article_id': '303',
  'source_title': 'Alabama',
  'source_url': 'https://en.wikipedia.org/wiki/Alabama',
  'evidence_quote': "Alabama's capital is Montgomery",
  'gold_passage_ids': ['303:0', '303:1']},
 {'id': 'q03',
  'question': 'In which city was Aristotle born?',
  'answerable': True,
  'answers': ['Stagira'],
  'article_id': '308',
  'source_title': 'Aristotle',
  'source_url': 'https://en.wikipedia.org/wiki/Aristotle',
  'evidence_quote': 'He was born in the city of Stagira in northern Greece '
                    'during the Classical period',
  'gold_passage_ids': ['308:0']},
 {'id': 'q04',
  'question': 'What philosophical system did Ayn Rand develop?',
  'answerable': True,
  'answers': ['Objectivism'],
  'article_id': '339',
  'source_title': 'Ayn Rand',
  'source_url': 'https://en.wikipedia.org/wiki/Ayn%20Rand',
  'evidence_quote': 'developing a philosophical system she named Objectivism',
  'gold_passage_ids': ['339:0']},
 {'id': 'q05',
  'question': 'What is the capital of Algeria?',
  'answerable': True,
  'answers': ['Algiers'],
  'article_id': '358',
  'source_title': 'Algeria',
  'source_url': 'https://en.wikipedia.org/wiki/Algeria',
  'evidence_quote': 'The capital and largest city is Algiers',
  'gold_passage_ids': ['358:1']},
 {'id': 'q06',
  'question': 'How many code points does ASCII have?',
  'answerable': True,
  'answers': ['128'],
  'article_id': '586',
  'source_title': 'ASCII',
  'source_url': 'https://en.wikipedia.org/wiki/ASCII',
  'evidence_quote': 'ASCII has just 128 code points',
  'gold_passage_ids': ['586:0']},
 {'id': 'q07',
  'question': 'What is the official language of Andorra?',
  'answerable': True,
  'answers': ['Catalan'],
  'article_id': '600',
  'source_title': 'Andorra',
  'source_url': 'https://en.wikipedia.org/wiki/Andorra',
  'evidence_quote': 'The official language is Catalan',
  'gold_passage_ids': ['600:1', '600:2']},
 {'id': 'q08',
  'question': 'Who wrote Animal Farm?',
  'answerable': True,
  'answers': ['George Orwell', 'Orwell'],
  'article_id': '620',
  'source_title': 'Animal Farm',
  'source_url': 'https://en.wikipedia.org/wiki/Animal%20Farm',
  'evidence_quote': 'by George Orwell, first published in England on 17 August '
                    '1945',
  'gold_passage_ids': ['620:0']},
 {'id': 'q09',
  'question': 'What is the state capital of Alaska?',
  'answerable': True,
  'answers': ['Juneau'],
  'article_id': '624',
  'source_title': 'Alaska',
  'source_url': 'https://en.wikipedia.org/wiki/Alaska',
  'evidence_quote': 'The state capital of Juneau is the second-largest city in '
                    'the United States by area',
  'gold_passage_ids': ['624:1', '624:2']},
 {'id': 'q10',
  'question': 'In which English town was Aldous Huxley born?',
  'answerable': True,
  'answers': ['Godalming'],
  'article_id': '628',
  'source_title': 'Aldous Huxley',
  'source_url': 'https://en.wikipedia.org/wiki/Aldous%20Huxley',
  'evidence_quote': 'Huxley was born in Godalming, Surrey, England, in 1894',
  'gold_passage_ids': ['628:2']},
 {'id': 'q11',
  'question': 'Who piloted the Command Module Columbia during Apollo 11?',
  'answerable': True,
  'answers': ['Michael Collins', 'Collins'],
  'article_id': '662',
  'source_title': 'Apollo 11',
  'source_url': 'https://en.wikipedia.org/wiki/Apollo%2011',
  'evidence_quote': 'pilot Michael Collins flew the Command Module Columbia in '
                    'lunar orbit',
  'gold_passage_ids': ['662:1']},
 {'id': 'q12',
  'question': "Approximately what percentage of Earth's surface does the "
              'Atlantic Ocean cover?',
  'answerable': True,
  'answers': ['17%', '17 percent', '17'],
  'article_id': '698',
  'source_title': 'Atlantic Ocean',
  'source_url': 'https://en.wikipedia.org/wiki/Atlantic%20Ocean',
  'evidence_quote': "It covers approximately 17% of Earth's surface",
  'gold_passage_ids': ['698:0']},
 {'id': 'q13',
  'question': 'In which year did Albert Einstein receive the Nobel Prize in '
              'Physics?',
  'answerable': True,
  'answers': ['1921'],
  'article_id': '736',
  'source_title': 'Albert Einstein',
  'source_url': 'https://en.wikipedia.org/wiki/Albert%20Einstein',
  'evidence_quote': 'He received the 1921 Nobel Prize in Physics',
  'gold_passage_ids': ['736:1']},
 {'id': 'q14',
  'question': 'What is the capital of Albania?',
  'answerable': True,
  'answers': ['Tirana'],
  'article_id': '738',
  'source_title': 'Albania',
  'source_url': 'https://en.wikipedia.org/wiki/Albania',
  'evidence_quote': 'Tirana is the capital and largest city in the country',
  'gold_passage_ids': ['738:1']},
 {'id': 'q15',
  'question': 'What is the capital of Azerbaijan?',
  'answerable': True,
  'answers': ['Baku'],
  'article_id': '746',
  'source_title': 'Azerbaijan',
  'source_url': 'https://en.wikipedia.org/wiki/Azerbaijan',
  'evidence_quote': 'Baku is the capital and largest city',
  'gold_passage_ids': ['746:0']},
 {'id': 'q16',
  'question': 'On what date did Ankara become the new Turkish capital?',
  'answerable': True,
  'answers': ['29 October 1923', 'October 29, 1923'],
  'article_id': '802',
  'source_title': 'Ankara',
  'source_url': 'https://en.wikipedia.org/wiki/Ankara',
  'evidence_quote': 'Ankara became the new Turkish capital upon the '
                    'establishment of the Republic on 29 October 1923',
  'gold_passage_ids': ['802:2', '802:3']},
 {'id': 'q17',
  'question': 'In which Dutch province is Amsterdam located?',
  'answerable': True,
  'answers': ['North Holland'],
  'article_id': '844',
  'source_title': 'Amsterdam',
  'source_url': 'https://en.wikipedia.org/wiki/Amsterdam',
  'evidence_quote': 'Located in the Dutch province of North Holland',
  'gold_passage_ids': ['844:0']},
 {'id': 'q18',
  'question': 'What explosive is Alfred Nobel most famous for inventing?',
  'answerable': True,
  'answers': ['dynamite'],
  'article_id': '851',
  'source_title': 'Alfred Nobel',
  'source_url': 'https://en.wikipedia.org/wiki/Alfred%20Nobel',
  'evidence_quote': "Nobel's most famous invention was dynamite",
  'gold_passage_ids': ['851:0']},
 {'id': 'q19',
  'question': 'In which city was Alexander Graham Bell born?',
  'answerable': True,
  'answers': ['Edinburgh'],
  'article_id': '852',
  'source_title': 'Alexander Graham Bell',
  'source_url': 'https://en.wikipedia.org/wiki/Alexander%20Graham%20Bell',
  'evidence_quote': 'Bell was born in Edinburgh, Scotland, on March 3, 1847',
  'gold_passage_ids': ['852:2']},
 {'id': 'q20',
  'question': 'To which continent is the aardvark native?',
  'answerable': True,
  'answers': ['Africa'],
  'article_id': '680',
  'source_title': 'Aardvark',
  'source_url': 'https://en.wikipedia.org/wiki/Aardvark',
  'evidence_quote': 'nocturnal mammal native to Africa',
  'gold_passage_ids': ['680:0']},
 {'id': 'q21',
  'question': "What does Euclid's algorithm compute for two integers?",
  'answerable': True,
  'answers': ['greatest common divisor',
              'greatest common divisor (GCD)',
              'GCD'],
  'article_id': '775',
  'source_title': 'Algorithm',
  'source_url': 'https://en.wikipedia.org/wiki/Algorithm',
  'evidence_quote': 'an efficient method for computing the greatest common '
                    'divisor (GCD) of two integers',
  'gold_passage_ids': ['775:41']},
 {'id': 'q22',
  'question': 'What type of rocket launched Apollo 11?',
  'answerable': True,
  'answers': ['Saturn V', 'Saturn V rocket'],
  'article_id': '662',
  'source_title': 'Apollo 11',
  'source_url': 'https://en.wikipedia.org/wiki/Apollo%2011',
  'evidence_quote': 'Apollo 11 was launched by a Saturn V rocket',
  'gold_passage_ids': ['662:1']},
 {'id': 'q23',
  'question': "In which year was Alfred Nobel's dynamite patented?",
  'answerable': True,
  'answers': ['1867'],
  'article_id': '851',
  'source_title': 'Alfred Nobel',
  'source_url': 'https://en.wikipedia.org/wiki/Alfred%20Nobel',
  'evidence_quote': 'dynamite, an explosive using  nitroglycerin; it was '
                    'patented in 1867',
  'gold_passage_ids': ['851:0']},
 {'id': 'q24',
  'question': 'What albedo value corresponds to a body that reflects all '
              'incident radiation?',
  'answerable': True,
  'answers': ['1', 'one'],
  'article_id': '39',
  'source_title': 'Albedo',
  'source_url': 'https://en.wikipedia.org/wiki/Albedo',
  'evidence_quote': '1 (corresponding to a body that reflects all incident '
                    'radiation',
  'gold_passage_ids': ['39:0']},
 {'id': 'q25',
  'question': 'Who won the Nobel Prize in Physics in 2024?',
  'answerable': False,
  'answers': [],
  'article_id': None,
  'source_title': None,
  'source_url': None,
  'evidence_quote': None,
  'gold_passage_ids': [],
  'unanswerable_reason': 'The question asks about an event after the November '
                         '2023 corpus snapshot.'},
 {'id': 'q26',
  'question': 'Which film won the Academy Award for Best Picture in 2024?',
  'answerable': False,
  'answers': [],
  'article_id': None,
  'source_title': None,
  'source_url': None,
  'evidence_quote': None,
  'gold_passage_ids': [],
  'unanswerable_reason': 'The question asks about an event after the November '
                         '2023 corpus snapshot.'},
 {'id': 'q27',
  'question': "Who won the Wimbledon men's singles title in 2024?",
  'answerable': False,
  'answers': [],
  'article_id': None,
  'source_title': None,
  'source_url': None,
  'evidence_quote': None,
  'gold_passage_ids': [],
  'unanswerable_reason': 'The question asks about an event after the November '
                         '2023 corpus snapshot.'},
 {'id': 'q28',
  'question': "What was Apple Inc.'s total revenue for fiscal year 2025?",
  'answerable': False,
  'answers': [],
  'article_id': None,
  'source_title': None,
  'source_url': None,
  'evidence_quote': None,
  'gold_passage_ids': [],
  'unanswerable_reason': 'The question asks about an event after the November '
                         '2023 corpus snapshot.'}]


In [46]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

def write_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temp.replace(path)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


In [47]:
def validate_inventory(spec):
    names = set()
    for entry in spec["files"]:
        name = PurePosixPath(entry["path"])
        if not name.parts or name.is_absolute() or ".." in name.parts or entry["path"] in names:
            raise ValueError("Invalid or duplicate inventory path")
        if not isinstance(entry["size"], int) or entry["size"] < 0:
            raise ValueError("Invalid inventory file size")
        if not (entry.get("sha256") or entry.get("git_sha1")):
            raise ValueError("Inventory entry lacks a content hash")
        names.add(entry["path"])
    if not names or sum(e["size"] for e in spec["files"]) != spec["total_bytes"]:
        raise ValueError("Incomplete inventory")
    return spec


def dataset_inventory():
    spec = validate_inventory(copy.deepcopy(EMBEDDED_INVENTORY["dataset"]))
    if (spec["repo_id"], spec["repo_type"], spec["revision"]) != (
        DATASET, "dataset", DATASET_REVISION
    ):
        raise ValueError("Expected the pinned wikimedia/wikipedia repository")
    shards = [entry for entry in spec["files"] if entry["path"].endswith(".parquet")]
    expected = {f"{CONFIG}/train-{i:05d}-of-00041.parquet" for i in range(41)}
    if {entry["path"] for entry in shards} != expected:
        raise ValueError("The inventory must contain all 41 English Parquet shards")
    if sum(entry["size"] for entry in shards) != 11630929031:
        raise ValueError("English download size does not match the pinned revision")
    if any("/" in entry["path"] and not entry["path"].startswith(CONFIG + "/")
           for entry in spec["files"]):
        raise ValueError("Non-English dataset paths are forbidden")
    return spec


In [48]:
def _file_valid(path: Path, entry: dict, verify_hashes: bool) -> bool:
    if not path.is_file() or path.stat().st_size != entry["size"]:
        return False
    if not verify_hashes:
        return True
    if "sha256" in entry:
        return sha256(path) == entry["sha256"]
    digest = hashlib.sha1(f'blob {entry["size"]}\0'.encode())
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest() == entry["git_sha1"]

def snapshot_status(destination: Path, spec: dict, verify_hashes=False) -> dict:
    missing = [entry for entry in spec["files"]
               if not _file_valid(destination / entry["path"], entry, verify_hashes)]
    return {
        "repo_id": spec["repo_id"], "repo_type": spec["repo_type"], "revision": spec["revision"],
        "total_files": len(spec["files"]), "valid_files": len(spec["files"]) - len(missing),
        "total_bytes": spec["total_bytes"], "missing_bytes": sum(e["size"] for e in missing),
        "missing": missing, "complete": not missing,
    }


In [49]:
def _download_files(destination: Path, spec: dict, entries: list[dict], max_workers: int):
    # A fresh process can download even if an earlier notebook cell imported Hub
    # with HF_HUB_OFFLINE=1. The parent kernel's offline settings remain unchanged.
    payload = {
        "repo_id": spec["repo_id"], "repo_type": spec["repo_type"], "revision": spec["revision"],
        "local_dir": str(destination.resolve()), "max_workers": max_workers,
        "missing": [e["path"] for e in entries if not (destination / e["path"]).exists()],
        "repair": [e["path"] for e in entries if (destination / e["path"]).exists()],
    }
    program = '''
import json, sys
from huggingface_hub import snapshot_download
p = json.load(sys.stdin)
for group, force in (("missing", False), ("repair", True)):
    if p[group]:
        snapshot_download(
            repo_id=p["repo_id"], repo_type=p["repo_type"], revision=p["revision"],
            local_dir=p["local_dir"], allow_patterns=p[group],
            force_download=force, max_workers=p["max_workers"],
        )
'''
    environment = os.environ.copy()
    for name in ("HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "TRANSFORMERS_OFFLINE"):
        environment[name] = "0"
    environment["HF_HUB_DISABLE_TELEMETRY"] = "1"
    environment["HF_XET_CACHE"] = str(destination / ".cache/huggingface/xet")
    with subprocess.Popen(
        [sys.executable, "-u", "-c", program], stdin=subprocess.PIPE,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment,
    ) as process:
        try:
            process.stdin.write(json.dumps(payload))
            process.stdin.close()
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait() != 0:
                raise RuntimeError("Dataset download interrupted or failed. Rerun to resume.")
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise


In [50]:
def ensure_local_snapshot(spec, destination, *, allow_download=True,
                          verify_hashes=False, max_workers=4):
    """Reuse a complete local copy; download/repair only invalid or missing entries."""
    validate_inventory(spec)
    destination = Path(destination)
    if max_workers < 1:
        raise ValueError("max_workers must be positive")
    status = snapshot_status(destination, spec, verify_hashes)
    if status["complete"]:
        print(f'{spec["repo_id"]}: {status["total_files"]} local files complete. '
              'Download skipped; no network needed.', flush=True)
        return destination
    if not allow_download:
        raise FileNotFoundError(
            f'{spec["repo_id"]}: {status["valid_files"]}/{status["total_files"]} '
            f'files present at {destination}. Enable downloads once while online.'
        )
    destination.mkdir(parents=True, exist_ok=True)
    (destination / ".download_complete.json").unlink(missing_ok=True)
    free = shutil.disk_usage(destination).free
    if free < status["missing_bytes"]:
        raise OSError(f'Need up to {status["missing_bytes"]/1e9:.2f} GB additional free space; '
                      f'{free/1e9:.2f} GB available. Choose another destination.')
    print(f'{spec["repo_id"]}: {status["valid_files"]}/{status["total_files"]} files complete; '
          f'downloading/resuming {len(status["missing"])} files.', flush=True)
    _download_files(destination, spec, status["missing"], max_workers)
    checked = snapshot_status(destination, spec, verify_hashes)
    if not checked["complete"]:
        raise RuntimeError(f'{len(checked["missing"])} files still incomplete; rerun to resume.')
    write_json(destination / ".download_complete.json", {
        key: checked[key] for key in ("repo_id", "revision", "total_files", "total_bytes", "complete")
    })
    print(f'{spec["repo_id"]}: all expected files are stored locally.', flush=True)
    return destination


def ensure_english_wikipedia(destination, *, allow_download=True, verify_hashes=False, max_workers=4):
    # Every English shard is included regardless of the indexing article count.
    return ensure_local_snapshot(dataset_inventory(), destination,
                                 allow_download=allow_download,
                                 verify_hashes=verify_hashes, max_workers=max_workers)


In [51]:
def offline_mode() -> None:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [52]:
inventory = dataset_inventory()
DATASET_DIR = ensure_local_snapshot(inventory, DATASET_DIR, allow_download=ALLOW_DOWNLOADS,
    verify_hashes=VERIFY_DATASET_HASHES, max_workers=DOWNLOAD_WORKERS)
MODEL_DIR = ensure_local_snapshot(EMBEDDED_INVENTORY['model'], MODEL_DIR,
    allow_download=ALLOW_DOWNLOADS, verify_hashes=True, max_workers=DOWNLOAD_WORKERS)
offline_mode()


wikimedia/wikipedia: 43 local files complete. Download skipped; no network needed.
google/flan-t5-small: 8 local files complete. Download skipped; no network needed.


In [53]:
UNKNOWN = "I don't know based on the provided evidence."
WORD = re.compile(r"\w+", re.UNICODE)
STOPWORDS = set(
    "a an the and or but of to in on at for from by with as is are was were be "
    "been being do does did has have had what which who whom whose when where "
    "why how can could would should will may might that this these those it its "
    "into about according many much name tell me give".split()
)


In [54]:
def span(text: str, candidate: str) -> str | None:
    """Return an original-text span, matching complete casefolded word tokens."""
    needles = [m.group().casefold() for m in WORD.finditer(candidate)]
    if not needles:
        return None
    matches = list(WORD.finditer(text))
    haystack = [m.group().casefold() for m in matches]
    for i in range(len(haystack) - len(needles) + 1):
        if haystack[i : i + len(needles)] == needles:
            return text[matches[i].start() : matches[i + len(needles) - 1].end()]
    return None

def is_unknown(text: str) -> bool:
    normalized = " ".join(WORD.findall(text.casefold()))
    return not normalized or normalized in {
        "unknown", "unanswerable", "not enough information", "not provided",
        "i don t know", "i do not know", "no answer", "not in the context",
        "i don t know based on the provided evidence",
    }

def passages(article: dict, chunk_words: int = 100, overlap: int = 25):
    if chunk_words < 1 or overlap < 0 or overlap >= chunk_words:
        raise ValueError("Require chunk_words > overlap >= 0.")
    text = article["text"]
    words = list(re.finditer(r"\S+", text))
    for ordinal, start in enumerate(range(0, len(words), chunk_words - overlap)):
        end = min(start + chunk_words, len(words))
        char_start, char_end = words[start].start(), words[end - 1].end()
        yield {
            "passage_id": f'{article["id"]}:{ordinal}',
            "article_id": str(article["id"]),
            "title": article["title"],
            "url": article["url"],
            "text": text[char_start:char_end],
            "char_start": char_start,
            "char_end": char_end,
        }
        if end == len(words):
            break


## Resumable full-corpus index
Passages retain title, URL, article ID and original character offsets. SQLite external-content FTS5 avoids storing passage text twice. Each transaction commits passage rows, search postings and its batch checkpoint together. Configuration and source file identity are checked on resume. A failed transaction is rolled back; completed batches are skipped. Never point this at the old small index.

The final audit compares processed article counts to Parquet metadata for **every shard**, verifies the FTS index against its content, and only then marks the index complete. Free-space checks are a stopping guard, not a prediction of required storage.


In [55]:
def build_full_index(snapshot, destination, chunk_words=100, overlap=25, batch_size=128):
    import pyarrow.parquet as pq
    if not (chunk_words > overlap >= 0 and batch_size > 0):
        raise ValueError("Invalid chunk or batch settings")
    files = sorted((snapshot / CONFIG).glob("train-*.parquet"))
    expected_names = [f"train-{i:05d}-of-00041.parquet" for i in range(41)]
    if [p.name for p in files] != expected_names:
        raise RuntimeError("All 41 English shards are required")
    signature = dict(schema=1, revision=DATASET_REVISION, chunk_words=chunk_words,
                     overlap=overlap, batch_size=batch_size,
                     files=[(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in files])
    signature = json.loads(json.dumps(signature))
    destination.parent.mkdir(parents=True, exist_ok=True)
    db = sqlite3.connect(destination)
    try:
        db.execute('PRAGMA journal_mode=WAL')
        db.execute('PRAGMA synchronous=FULL')
        db.execute('PRAGMA cache_size=-65536')
        db.execute('PRAGMA temp_store=FILE')
        db.executescript("""
        CREATE TABLE IF NOT EXISTS metadata(key TEXT PRIMARY KEY, value TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS batches(shard TEXT, batch INTEGER, articles INTEGER,
            chunks INTEGER, PRIMARY KEY(shard,batch));
        CREATE TABLE IF NOT EXISTS passages(
            passage_id TEXT UNIQUE, article_id TEXT, title TEXT, url TEXT, text TEXT,
            char_start INTEGER, char_end INTEGER);
        CREATE VIRTUAL TABLE IF NOT EXISTS search USING fts5(title,text,
            content='passages',content_rowid='rowid',tokenize='porter unicode61');
        """)
        def get(key):
            r = db.execute('SELECT value FROM metadata WHERE key=?',(key,)).fetchone()
            return json.loads(r[0]) if r else None
        def put(key,value):
            db.execute('INSERT OR REPLACE INTO metadata VALUES (?,?)',(key,json.dumps(value)))
        old = get('signature')
        if old is not None and old != signature:
            raise RuntimeError('Index/source settings changed. Use a new INDEX_PATH.')
        with db: put('signature',signature)
        if get('complete'):
            return get('audit')
        total_expected = total_processed = total_chunks = 0
        for path in files:
            parquet = pq.ParquetFile(path)
            expected = parquet.metadata.num_rows
            total_expected += expected
            done = {r[0] for r in db.execute('SELECT batch FROM batches WHERE shard=?',(path.name,))}
            for batch_id, batch in enumerate(parquet.iter_batches(batch_size=batch_size,
                    columns=['id','title','url','text'], use_threads=False)):
                if batch_id in done:
                    continue
                if shutil.disk_usage(destination.parent).free < MIN_FREE_GB * 10**9:
                    raise RuntimeError('Low disk space. Free space and rerun to resume.')
                chunks = 0
                with db:
                    for article in batch.to_pylist():
                        for p in passages(article,chunk_words,overlap):
                            cur = db.execute('INSERT INTO passages VALUES (?,?,?,?,?,?,?)',
                                tuple(p[k] for k in ['passage_id','article_id','title','url','text','char_start','char_end']))
                            db.execute('INSERT INTO search(rowid,title,text) VALUES (?,?,?)',
                                       (cur.lastrowid,p['title'],p['text']))
                            chunks += 1
                    db.execute('INSERT INTO batches VALUES (?,?,?,?)',
                               (path.name,batch_id,batch.num_rows,chunks))
                if batch_id % 100 == 0:
                    print(path.name, 'batch',batch_id, 'index GB',round(destination.stat().st_size/1e9,2), flush=True)
            count, chunks = db.execute('SELECT coalesce(sum(articles),0),coalesce(sum(chunks),0) FROM batches WHERE shard=?',
                                      (path.name,)).fetchone()
            if count != expected:
                raise RuntimeError(f'Incomplete shard: {path.name}: {count}/{expected}')
            total_processed += count
            total_chunks += chunks
            db.execute('PRAGMA wal_checkpoint(TRUNCATE)')
            print('Complete:',path.name, count,'articles',flush=True)
        assert total_processed == total_expected
        assert db.execute('SELECT count(*) FROM passages').fetchone()[0] == total_chunks
        # rank=1 additionally checks external-content rows against the inverted index.
        with db:
            db.execute("INSERT INTO search(search,rank) VALUES ('integrity-check',1)")
            audit = dict(articles=total_processed, expected_articles=total_expected,
                         passages=total_chunks, shards=len(files), complete=True)
            put('audit',audit)
            put('complete',True)
        db.execute('PRAGMA wal_checkpoint(TRUNCATE)')
        return audit
    finally:
        db.close()


In [56]:
# Long-running first build; subsequent runs validate the signature and reuse the index.
audit = build_full_index(DATASET_DIR, INDEX_PATH, CHUNK_WORDS, OVERLAP, BATCH_SIZE)
print(json.dumps(audit, indent=2))


{
  "articles": 6407814,
  "expected_articles": 6407814,
  "passages": 42518633,
  "shards": 41,
  "complete": true
}


In [57]:
class Retriever:
    def __init__(self, path: Path):
        if not path.is_file():
            raise FileNotFoundError(f"Index missing: {path}. Run the index command first.")
        self.connection = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)
        self.connection.row_factory = sqlite3.Row
        self.metadata = {
            row["key"]: json.loads(row["value"])
            for row in self.connection.execute("SELECT key,value FROM metadata")
        }

        if not self.metadata.get('complete'):
            self.connection.close()
            raise RuntimeError('Full corpus build is incomplete. Resume the indexing cell.')

    def close(self):
        self.connection.close()

    def search(self, question: str, k: int = 3) -> list[dict]:
        if k < 0:
            raise ValueError("k must be nonnegative.")
        terms = list(dict.fromkeys(
            token.casefold() for token in WORD.findall(question)
            if token.casefold() not in STOPWORDS
        ))[:48]
        if not terms or k == 0:
            return []
        # Quoted tokens and bound parameters prevent FTS query syntax injection.
        query = " OR ".join(f'"{term}"' for term in terms)
        rows = self.connection.execute("""
            SELECT p.*, bm25(search,2.0,1.0) AS bm25
            FROM search JOIN passages p ON p.rowid=search.rowid
            WHERE search MATCH ? ORDER BY bm25 ASC, p.rowid ASC LIMIT ?
        """, (query, k))
        return [dict(row) for row in rows]


## Offline generation and evidence fusion
`concatenate` fairly packs the top-k passages into one 512-token prompt. `generate_from_evidence` generates a candidate from each passage, retains only candidates found in that visible passage, then answers from their original evidence sentences. The final answer must also match visible original evidence. Both methods expose full traces, retrieved passages, token counts and citations. Larger k shrinks each excerpt in concatenate; the evidence method costs up to k+1 model calls. Neither is FiD (decoder-level fusion).


In [58]:
class Generator:
    def __init__(self, model_dir: Path | None = None, device="cpu", threads=4,
                 max_input_tokens=512, max_new_tokens=48):
        offline_mode()
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        model_dir = model_dir or ROOT / "models" / "flan-t5-small"
        if not (model_dir / "model.safetensors").exists():
            raise FileNotFoundError(f"Model missing at {model_dir}. Run prepare while online.")
        if threads < 1 or max_new_tokens < 1 or not 64 <= max_input_tokens <= 512:
            raise ValueError("Require positive threads/output tokens and input budget 64..512.")
        torch.set_num_threads(threads)
        torch.manual_seed(42)
        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_dir, local_files_only=True, use_safetensors=True,
        ).to(device).eval()
        self.device = device
        self.max_input_tokens = max_input_tokens
        self.max_new_tokens = max_new_tokens
        self.threads = threads

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=True, verbose=False))

    @staticmethod
    def render(question: str, evidence: list[dict], stage="answer") -> str:
        if stage == "closed_book":
            return f"Answer the question briefly.\nQuestion: {question}\nAnswer:"
        instruction = (
            "Copy the shortest phrase from the context that answers the question."
            if stage == "extract"
            else "Answer the question briefly using only the context."
        )
        context = "\n\n".join(
            f'[{item["passage_id"]}] {item["title"]}\n{item["text"]}' for item in evidence
        )
        return (
            f"{instruction} If the context does not contain the answer, say unknown.\n"
            f"Context:\n{context}\nQuestion: {question}\nAnswer:"
        )

    def pack(self, question: str, evidence: list[dict], stage="answer"):
        """Fairly allocate a fixed input budget, preserving whole words and headers."""
        question = question.strip()
        if not question:
            raise ValueError("Question cannot be empty.")
        words = [re.findall(r"\S+", item["text"]) for item in evidence]

        def make(lengths):
            visible = [
                {key: item[key] for key in ("passage_id", "title", "url")}
                | {"text": " ".join(tokens[:length])}
                for item, tokens, length in zip(evidence, words, lengths)
            ]
            prompt = self.render(question, visible, stage)
            return prompt, visible, self.count_tokens(prompt)

        if not evidence:
            prompt, visible, size = make([])
            if size > self.max_input_tokens:
                raise ValueError("Question exceeds the model input budget.")
            return prompt, visible, size
        _, _, overhead = make([0] * len(words))
        if overhead > self.max_input_tokens - 16:
            raise ValueError("Question/source headers leave too little evidence space; reduce k.")
        low, high = 0, max(map(len, words))
        while low < high:
            middle = (low + high + 1) // 2
            if make([min(middle, len(tokens)) for tokens in words])[2] <= self.max_input_tokens:
                low = middle
            else:
                high = middle - 1
        lengths = [min(low, len(tokens)) for tokens in words]
        # Fill remaining capacity in retrieval order, never truncate the prompt.
        changed = True
        while changed:
            changed = False
            for i, tokens in enumerate(words):
                if lengths[i] < len(tokens):
                    trial = lengths.copy()
                    trial[i] += 1
                    if make(trial)[2] <= self.max_input_tokens:
                        lengths = trial
                        changed = True
        return make(lengths)

    def generate(self, prompt: str) -> dict:
        encoded = self.tokenizer(prompt, return_tensors="pt", truncation=False).to(self.device)
        input_tokens = int(encoded.input_ids.shape[1])
        if input_tokens > self.max_input_tokens:
            raise ValueError(f"Input has {input_tokens} tokens; budget is {self.max_input_tokens}.")
        start = time.perf_counter()
        with self.torch.inference_mode():
            output = self.model.generate(
                **encoded, do_sample=False, num_beams=1,
                max_new_tokens=self.max_new_tokens,
            )
        elapsed = time.perf_counter() - start
        return {
            "text": self.tokenizer.decode(output[0], skip_special_tokens=True).strip(),
            "input_tokens": input_tokens, "output_tokens": int(output.shape[1] - 1),
            "generation_seconds": elapsed,
        }


In [59]:
def supporting_sources(answer: str, visible: list[dict], originals: list[dict]) -> list[dict]:
    if is_unknown(answer):
        return []
    lookup = {item["passage_id"]: item for item in originals}
    sources = []
    for item in visible:
        original = lookup[item["passage_id"]]
        # Check both the model-visible excerpt and the original passage.
        if span(item["text"], answer) is not None:
            quote = span(original["text"], answer)
            if quote is not None:
                sources.append({
                    "passage_id": item["passage_id"], "title": original["title"],
                    "url": original["url"], "answer_span": quote,
                    "visible_excerpt": item["text"],
                    "char_start": original["char_start"], "char_end": original["char_end"],
                })
    return sources

def evidence_sentence(text: str, candidate: str) -> str:
    """Keep the candidate's original sentence so the reduce stage retains its relation."""
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if span(sentence, candidate) is not None:
            return sentence.strip()
    # A generated span can cross sentence boundaries; retain the visible passage.
    return text


In [60]:
def answer_question(generator: Generator, question: str, retrieved: list[dict],
                    method="concatenate") -> dict:
    if method not in {"concatenate", "generate_from_evidence", "closed_book"}:
        raise ValueError(f"Unknown method: {method}")
    start = time.perf_counter()
    trace, map_steps = [], []
    if method == "closed_book":
        prompt, visible, _ = generator.pack(question, [], stage="closed_book")
        generated = generator.generate(prompt)
        trace.append({"stage": "closed_book", "prompt": prompt, **generated})
        raw_answer = generated["text"]
        sources = []
        answer = UNKNOWN if is_unknown(raw_answer) else raw_answer
    elif not retrieved:
        raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    else:
        evidence = retrieved
        if method == "generate_from_evidence":
            evidence = []
            for passage in retrieved:
                prompt, map_visible, _ = generator.pack(question, [passage], stage="extract")
                generated = generator.generate(prompt)
                trace.append({"stage": "extract", "prompt": prompt, **generated})
                candidate = generated["text"]
                supported = supporting_sources(candidate, map_visible, [passage])
                map_steps.append({
                    "passage_id": passage["passage_id"], "candidate": candidate,
                    "accepted": bool(supported), "visible_evidence": map_visible,
                })
                if supported:
                    # The reduce stage receives original quotes, never unverified model claims.
                    quote = evidence_sentence(map_visible[0]["text"], candidate)
                    map_steps[-1]["evidence_sentence"] = quote
                    evidence.append(passage | {"text": quote})
        if evidence:
            prompt, visible, _ = generator.pack(question, evidence)
            generated = generator.generate(prompt)
            trace.append({"stage": "answer", "prompt": prompt, **generated})
            raw_answer = generated["text"]
            sources = supporting_sources(raw_answer, visible, retrieved)
            answer = raw_answer if sources else UNKNOWN
        else:
            raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    return {
        "question": question, "method": method, "requested_passages_used": len(retrieved),
        "answer": answer, "raw_answer": raw_answer, "abstained": answer == UNKNOWN,
        "span_supported": bool(sources), "citations": sources, "retrieved": retrieved,
        "visible_evidence": visible, "map_steps": map_steps, "trace": trace,
        "generation_calls": len(trace),
        "total_input_tokens": sum(item["input_tokens"] for item in trace),
        "total_output_tokens": sum(item["output_tokens"] for item in trace),
        "latency_seconds": time.perf_counter() - start,
        "grounding_policy": "contiguous lexical span in visible and original evidence"
        if method != "closed_book" else "none (closed-book baseline)",
    }


In [61]:
if 'retriever' in globals():
    retriever.close()
retriever = Retriever(INDEX_PATH)
generator = Generator(MODEL_DIR, device=DEVICE, threads=THREADS,
                      max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS)
def ask(question, k=3, method='generate_from_evidence'):
    result = answer_question(generator, question, retriever.search(question,k),method)
    print(result['answer'])
    for source in result['citations']:
        print(source['title'], source['url'], '\nEvidence:', source['visible_excerpt'])
    return result
result = ask(QUESTION, K)


-Aluminium
Aluminium https://en.wikipedia.org/wiki/Aluminium 
Evidence: the three heavier trihalides, aluminium fluoride (AlF3) features six-coordinate aluminium, which explains its involatility and insolubility as well as high heat of formation.
Coordination number https://en.wikipedia.org/wiki/Coordination%20number 
Evidence: α-Aluminium has a regular cubic close packed structure, fcc, where each aluminium atom has 12 nearest neighbors, 6 in the same plane and 3 above and below and the coordination polyhedron is a cuboctahedron.


## Retrieval-size and fusion ablations
Run k=1,3,5 for both fusion methods plus a closed-book baseline. The inherited 28 questions are a **development smoke benchmark**, biased toward early Wikipedia articles; their old unanswerable labels are not assumed valid for the full corpus and those items are excluded. These numbers must not be presented as corpus-wide accuracy. Supply an independent, manually verified JSONL benchmark covering topics across all shards for a meaningful generalization assessment. Fields: `id`, `question`, `answers` (nonempty alias list), `answerable: true`, `article_id`, `gold_passage_ids`, `evidence_quote`. The validator checks provenance against the full index.

Reports include exact match, token F1, gold-passage hit@k, abstention, lexical support, latency and model calls. Lexical support is not a semantic faithfulness score. A single nested top-max(k) retrieval is reused across methods for each question. Full traces are saved for audit.


In [62]:
def normalize(text: str) -> str:
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", text).split())

def scores(prediction: str, aliases: list[str]) -> tuple[float, float]:
    prediction = normalize(prediction)
    em, f1 = 0.0, 0.0
    for alias in aliases:
        gold = normalize(alias)
        em = max(em, float(prediction == gold))
        predicted_tokens, gold_tokens = prediction.split(), gold.split()
        overlap = sum((Counter(predicted_tokens) & Counter(gold_tokens)).values())
        if overlap:
            f1 = max(f1, 2 * overlap / (len(predicted_tokens) + len(gold_tokens)))
    return em, f1

def validate_questions(questions, retriever):
    ids = set()
    if not questions:
        raise ValueError("The evaluation set is empty.")
    for question in questions:
        if question["id"] in ids:
            raise ValueError(f'Duplicate evaluation id: {question["id"]}')
        ids.add(question["id"])
        if not question["question"].strip():
            raise ValueError("Empty question.")
        if question["answerable"]:
            if not question["answers"] or not question["gold_passage_ids"]:
                raise ValueError(f'Missing gold answer/provenance for {question["id"]}')
            for passage_id in question["gold_passage_ids"]:
                row = retriever.connection.execute(
                    "SELECT text, article_id FROM passages WHERE passage_id=?", (passage_id,),
                ).fetchone()
                if row is None or row["article_id"] != question["article_id"]:
                    raise ValueError(f"Gold passage not in this index: {passage_id}")
                if not span(row["text"], question["evidence_quote"]):
                    raise ValueError(f"Gold evidence mismatch: {passage_id}")
                if not any(span(row["text"], alias) for alias in question["answers"]):
                    raise ValueError(f"Gold answer absent from passage: {passage_id}")
        elif question["answers"] or question["gold_passage_ids"]:
            raise ValueError("Unanswerable items must have empty answers and gold_passage_ids.")

def mean(rows, key):
    values = [row[key] for row in rows if row.get(key) is not None]
    return statistics.mean(values) if values else None


In [63]:
EVAL_JSONL = None  # Optional independent benchmark path.
questions = (read_jsonl(Path(EVAL_JSONL)) if EVAL_JSONL else
             [q for q in EVAL_QUESTIONS if q['answerable']])
validate_questions(questions, retriever)
result_dir = ROOT / 'results' / ('full_wikipedia_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
result_dir.mkdir(parents=True, exist_ok=True)
if RUN_FULL_ABLATION:
    rows = []
    with (result_dir / 'predictions.jsonl').open('w') as out:
        for q in questions:
            start = time.perf_counter()
            hits = retriever.search(q['question'], max(KS))
            retrieval_seconds = time.perf_counter()-start
            for method, k in [('closed_book',0)] + [(m,k) for m in ['concatenate','generate_from_evidence'] for k in KS]:
                result = answer_question(generator,q['question'],hits[:k],method)
                em,f1 = scores(result['answer'],q['answers'])
                row = dict(id=q['id'],method=method,k=k,em=em,f1=f1,
                    gold_passage_hit=float(bool(set(q['gold_passage_ids']) & {p['passage_id'] for p in hits[:k]})) if k else None,
                    abstention=float(result['abstained']),span_support=float(result['span_supported']) if k else None,
                    latency_seconds=result['latency_seconds']+(retrieval_seconds if k else 0),
                    generation_calls=result['generation_calls'])
                rows.append(row)
                out.write(json.dumps(dict(metrics=row,result=result))+'\n')
                out.flush()
    frame = pd.DataFrame(rows)
    summary = frame.groupby(['method','k']).mean(numeric_only=True).reset_index()
    frame.to_csv(result_dir/'metrics.csv',index=False)
    summary.to_csv(result_dir/'summary.csv',index=False)
    write_json(result_dir/'manifest.json',dict(index_audit=audit,dataset_revision=DATASET_REVISION,
        model_revision=MODEL_REVISION,ks=KS,benchmark='custom' if EVAL_JSONL else 'development-only',
        question_count=len(questions),python=sys.version,
        packages={p:importlib.metadata.version(p) for p in ['torch','transformers','pyarrow']},
        chunk_words=CHUNK_WORDS,overlap=OVERLAP,input_tokens=MAX_INPUT_TOKENS,device=DEVICE))
    display(summary)
    print('Saved:',result_dir)


## Attribution and limitations
Wikipedia text remains attributed through article titles and URLs. Chunking creates excerpts with retained character offsets. See the [dataset card](https://huggingface.co/datasets/wikimedia/wikipedia) for CC-BY-SA 3.0 / GFDL licensing, and the [Flan-T5-small model card](https://huggingface.co/google/flan-t5-small) for Apache 2.0 and model limitations. Index design follows [SQLite FTS5](https://www.sqlite.org/fts5.html). The full dataset is already downloaded locally; building the full index is a separate operation. No full-corpus results are precomputed or claimed in this notebook.
